# Day 15 — Exercises: Deep Data Cleaning

Complete the following exercises to practice the core concepts for Day 15.

**Topics covered:**
- Outliers detection and handling (IQR & Z-score)
- Duplicate detection and handling (Exact & Key-based)
- Category standardization (Case, whitespace, mapping)
- Type coercion (Safely handling numeric and date formatting issues)
- Encoding categorical values (One-hot & Ordinal mapping)

## Exercise 1: Outliers (IQR & Z-score)

In this exercise, you will write functions to detect and handle outliers in a dataset containing salary information.

### Tasks:
1. Compute the lower and upper bounds of a column using the IQR method ($Q1 - 1.5 \times IQR$ and $Q3 + 1.5 \times IQR$).
2. Filter rows that contain outliers based on the IQR bounds.
3. Compute outliers using the Z-score method (where $|Z| > 3$ standard deviations from the mean).
4. Implement a capping function (Winsorization) to clip outliers to the upper and lower bounds instead of deleting them.

In [ ]:
import pandas as pd
import numpy as np

# Create sample data with outliers
np.random.seed(42)
salaries = np.random.normal(50000, 10000, 100)
# Manually inject extreme outliers
salaries[10] = 180000
salaries[35] = -25000
salaries[75] = 220000

df = pd.DataFrame({'Salary': salaries})
print(f"Original shape: {df.shape}")

# 1. Calculate IQR bounds
q1 = df['Salary'].quantile(0.25)
q3 = df['Salary'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"IQR Bounds: Lower = {lower_bound:.2f}, Upper = {upper_bound:.2f}")

# 2. Filter outliers using IQR
df_iqr_cleaned = df[(df['Salary'] >= lower_bound) & 
                    (df['Salary'] <= upper_bound)]
print(f"Shape after IQR outlier removal: {df_iqr_cleaned.shape}")

# 3. Identify outliers using Z-score (threshold > 3)
mean_val = df['Salary'].mean()
std_val = df['Salary'].std()
z_scores = (df['Salary'] - mean_val) / std_val
df_z_cleaned = df[z_scores.abs() <= 3]
print(f"Shape after Z-score outlier removal: {df_z_cleaned.shape}")

# 4. Cap outliers (Winsorization) using pandas clip
df_capped = df.copy()
df_capped['Salary'] = df_capped['Salary'].clip(lower_bound, upper_bound)
print(f"Min value after capping: {df_capped['Salary'].min():.2f}")
print(f"Max value after capping: {df_capped['Salary'].max():.2f}")
print(f"Shape after capping: {df_capped.shape}")

Original shape: (100, 1)
IQR Bounds: Lower = 27464.57, Upper = 71534.90
Shape after IQR outlier removal: (96, 1)
Shape after Z-score outlier removal: (97, 1)
Min value after capping: 27464.57
Max value after capping: 71534.90
Shape after capping: (100, 1)


## Exercise 2: Duplicates

In this exercise, you will practice finding and removing both exact duplicates and key-based duplicates in a user activity dataset.

### Tasks:
1. Count and print the number of exact duplicate rows.
2. Remove exact duplicates, keeping the first occurrence.
3. Remove duplicates based on a subset of columns (e.g., `UserID` and `Date`), keeping the latest action (last occurrence).

In [7]:
data = {
    'UserID': [101, 102, 103, 101, 101, 104, 102],
    'Date': ['2026-06-01', '2026-06-01', '2026-06-02', '2026-06-01', '2026-06-01', '2026-06-02', '2026-06-01'],
    'Action': ['Login', 'Login', 'Logout', 'Login', 'Login', 'Login', 'Logout']
}
df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)
print()

# 1. Count exact duplicates
exact_dupes = df.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_dupes}")

# 2. Remove exact duplicates (keep first)
df_no_exact = df.drop_duplicates(keep='first')
print("\nDataFrame after removing exact duplicates:")
print(df_no_exact)

# 3. Remove duplicates based on UserID and Date, keeping the last occurrence (latest activity)
df_no_subset = df.drop_duplicates(subset=['UserID', 'Date'], keep='last')
print("\nDataFrame after subset deduplication (latest activity per User/Date):")
print(df_no_subset)

Original DataFrame:
   UserID        Date  Action
0     101  2026-06-01   Login
1     102  2026-06-01   Login
2     103  2026-06-02  Logout
3     101  2026-06-01   Login
4     101  2026-06-01   Login
5     104  2026-06-02   Login
6     102  2026-06-01  Logout

Number of exact duplicate rows: 2

DataFrame after removing exact duplicates:
   UserID        Date  Action
0     101  2026-06-01   Login
1     102  2026-06-01   Login
2     103  2026-06-02  Logout
5     104  2026-06-02   Login
6     102  2026-06-01  Logout

DataFrame after subset deduplication (latest activity per User/Date):
   UserID        Date  Action
2     103  2026-06-02  Logout
4     101  2026-06-01   Login
5     104  2026-06-02   Login
6     102  2026-06-01  Logout


## Exercise 3: Inconsistent Categories

Categorical columns often have spelling variations, leading/trailing whitespace, or casing issues that prevent proper grouping.

### Tasks:
1. Standardize string case (lowercase) and strip whitespace from the `Country` column.
2. Map inconsistent entries to standardized labels using a consolidation dictionary.

In [8]:
df = pd.DataFrame({
    'Country': [' United States ', 'usa', 'U.S.A.', 'united kingdom', 'uk', 'U.K. ', 'Germany', 'germany '],
    'Sales': [100, 200, 150, 120, 80, 90, 300, 250]
})
print("Unique countries before cleaning:")
print(df['Country'].unique())

# 1. Lowercase and strip whitespace
df['Country'] = df['Country'].str.lower().str.strip()
print("\nUnique countries after case and strip standardization:")
print(df['Country'].unique())

# 2. Consolidation mapping dictionary
country_map = {
    'united states': 'United States',
    'usa': 'United States',
    'u.s.a.': 'United States',
    'united kingdom': 'United Kingdom',
    'uk': 'United Kingdom',
    'u.k.': 'United Kingdom',
    'germany': 'Germany'
}
df['Country'] = df['Country'].map(country_map)

print("\nUnique countries after mapping consolidation:")
print(df['Country'].unique())
print("\nCleaned DataFrame:")
print(df)

Unique countries before cleaning:
<ArrowStringArray>
[' United States ',             'usa',          'U.S.A.',  'united kingdom',
              'uk',           'U.K. ',         'Germany',        'germany ']
Length: 8, dtype: str

Unique countries after case and strip standardization:
<ArrowStringArray>
['united states', 'usa', 'u.s.a.', 'united kingdom', 'uk', 'u.k.', 'germany']
Length: 7, dtype: str

Unique countries after mapping consolidation:
<ArrowStringArray>
['United States', 'United Kingdom', 'Germany']
Length: 3, dtype: str

Cleaned DataFrame:
          Country  Sales
0   United States    100
1   United States    200
2   United States    150
3  United Kingdom    120
4  United Kingdom     80
5  United Kingdom     90
6         Germany    300
7         Germany    250


## Exercise 4: Type Coercion

When importing raw files (like CSVs), numeric or datetime columns can import as 'object' types if they contain corrupt string placeholders (like 'N/A', 'NULL', or random text).

### Tasks:
1. Identify column types in a messy dataset.
2. Coerce columns to correct numeric types, converting unparseable items to `NaN`.
3. Coerce dates using `pd.to_datetime` with `errors='coerce'`.

In [9]:
df = pd.DataFrame({
    'Price': ['12.50', '14.20', 'CorruptText', '9.99', 'N/A'],
    'Date': ['2026-06-01', '2026-06-02', '2026-06-03', 'InvalidDate', '2026-06-05']
})
print("Column data types before coercion:")
print(df.dtypes)
print()

# 1. Coerce Price to numeric
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# 2. Coerce Date to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print("Column data types after coercion:")
print(df.dtypes)
print("\nDataFrame contents (corrupt elements replaced with NaT/NaN):")
print(df)

Column data types before coercion:
Price    str
Date     str
dtype: object

Column data types after coercion:
Price           float64
Date     datetime64[us]
dtype: object

DataFrame contents (corrupt elements replaced with NaT/NaN):
   Price       Date
0  12.50 2026-06-01
1  14.20 2026-06-02
2    NaN 2026-06-03
3   9.99        NaT
4    NaN 2026-06-05


## Exercise 5: Categorical Encoding

Convert standardized categorical values into a numerical format suitable for downstream ML modeling.

### Tasks:
1. Encode an ordered category (`Education`) using ordinal mapping.
2. Encode nominal categories (`City`) using one-hot encoding.

In [10]:
df = pd.DataFrame({
    'UserID': [1, 2, 3, 4, 5],
    'Education': ['High School', 'Bachelor', 'Master', 'PhD', 'Bachelor'],
    'City': ['New York', 'Los Angeles', 'Chicago', 'New York', 'Chicago']
})
print("Original DataFrame:")
print(df)
print()

# 1. Ordinal Encoding for Education column
education_order = {
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}
df['Education_Encoded'] = df['Education'].map(education_order)

# 2. One-hot Encoding for City column
df_encoded = pd.get_dummies(df, columns=['City'], drop_first=True)

print("Final Encoded DataFrame:")
print(df_encoded)

Original DataFrame:
   UserID    Education         City
0       1  High School     New York
1       2     Bachelor  Los Angeles
2       3       Master      Chicago
3       4          PhD     New York
4       5     Bachelor      Chicago

Final Encoded DataFrame:
   UserID    Education  Education_Encoded  City_Los Angeles  City_New York
0       1  High School                  1             False           True
1       2     Bachelor                  2              True          False
2       3       Master                  3             False          False
3       4          PhD                  4             False           True
4       5     Bachelor                  2             False          False
